# 02 — Tokenización: de caracteres a subpalabras

**LLM Lab — Etapa 1 · Fase 2 (Tokenización)**

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/USUARIO/llm-lab/blob/main/notebooks/02_tokenizacion.ipynb) *(placeholder)*

## Objetivos de aprendizaje

Al terminar este notebook serás capaz de:

1. **Explicar** por qué un LLM no puede operar sobre palabras crudas (problema del vocabulario abierto) y comparar los tres niveles de tokenización: caracteres, palabras y subpalabras.
2. **Derivar e implementar desde cero** el algoritmo BPE (Byte-Pair Encoding), incluyendo el conteo de pares y el merge iterativo, y trazar cómo evoluciona la segmentación de una palabra técnica a medida que crece el número de merges.
3. **Distinguir** las cuatro familias de tokenizadores modernos — BPE byte-level (GPT-2), WordPiece (BERT), SentencePiece y Unigram LM — derivando matemáticamente el criterio de cada una.
4. **Medir** la calidad de un tokenizador con métricas cuantitativas: fertilidad (tokens/palabra), compresión (chars/token, bytes/token), entropía de la distribución de tokens y cobertura de términos técnicos.
5. **Entrenar y guardar** un tokenizador ByteLevel-BPE propio con la librería `tokenizers` (Rust), con normalización NFKC y verificación round-trip, listo para las fases siguientes del lab.

## Prerrequisitos

- Notebook `00` (visión general) y `01` (corpus): qué es un corpus, los 21 dominios del lab.
- Python intermedio (dicts, comprehensions, `collections.Counter`).
- Probabilidad básica: $\log$, verosimilitud, entropía.

## Tiempo estimado

| Perfil | Hardware | Tiempo |
|--------|----------|--------|
| SMALL  | CPU / T4 (Colab Free) | ~45 min |
| LARGE  | A100 (Colab Pro) | ~60 min |

> La tokenización corre en CPU: la GPU solo se detecta para fijar el `PROFILE` automáticamente.

## Tabla de contenidos

1. [¿Por qué tokenizar? El problema del vocabulario abierto](#sec1)
2. [BPE (Byte-Pair Encoding) desde cero](#sec2)
3. [Byte-level BPE: el truco de GPT-2](#sec3)
4. [WordPiece: el criterio de verosimilitud de BERT](#sec4)
5. [SentencePiece: tokenizar sin pre-segmentar](#sec5)
6. [Unigram LM: tokenización probabilística](#sec6)
7. [Comparación de tokenizadores reales (GPT-2, BERT, Llama)](#sec7)
8. [Entrenar nuestro propio tokenizador con `tokenizers`](#sec8)
9. [Anatomía del vocabulario aprendido](#sec9)
10. [Frecuencia de tokens y ley de Zipf](#sec10)
11. [Compresión del lenguaje: chars/token y bytes/token](#sec11)
12. [Cobertura de términos técnicos](#sec12)
13. [Distribución de longitudes de secuencia](#sec13)
14. [Resumen, ejercicios y referencias](#sec14)

In [ ]:
# ── Celda de setup estándar del LLM Lab ──────────────────────────────
# Instala dependencias faltantes (sin reinstalar torch en Colab),
# detecta hardware, fija seeds y configura matplotlib/seaborn.
import importlib, subprocess, sys, os, random

def ensure(packages):
    """Instala con pip -q solo los paquetes que falten."""
    for pkg in packages:
        try:
            importlib.import_module(pkg)
        except ImportError:
            print("Instalando {} ...".format(pkg))
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg],
                           check=False)

ensure(["tokenizers", "transformers", "sentencepiece", "seaborn"])

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# ¿Estamos en Colab?
IN_COLAB = "google.colab" in sys.modules

# Detección de hardware: CUDA > MPS > CPU (torch es opcional aquí)
DEVICE, GPU_NAME, VRAM_GB = "cpu", None, 0.0
try:
    import torch
    if torch.cuda.is_available():
        DEVICE = "cuda"
        GPU_NAME = torch.cuda.get_device_name(0)
        VRAM_GB = torch.cuda.get_device_properties(0).total_memory / 1024**3
    elif getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():
        DEVICE = "mps"
        GPU_NAME = "Apple Silicon (MPS)"
except ImportError:
    torch = None

# Seeds reproducibles en todas las fuentes de aleatoriedad
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
if torch is not None:
    torch.manual_seed(SEED)
    if DEVICE == "cuda":
        torch.cuda.manual_seed_all(SEED)

# Estilo de gráficos del lab
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["axes.titlesize"] = 13

print("Colab: {} | Device: {} | GPU: {} | VRAM: {:.1f} GB | SEED: {}".format(
    IN_COLAB, DEVICE, GPU_NAME, VRAM_GB, SEED))

In [ ]:
# ── Configuración SMALL / LARGE ──────────────────────────────────────
# PROFILE = "auto" se resuelve por VRAM; puedes forzarlo a "small"/"large".
# Espeja configs/small.yaml y configs/large.yaml del repo.
PROFILE = "auto"   # "auto" | "small" | "large"

CONFIGS = {
    "small": {           # Colab Free (T4) / CPU — vocab del modelo de ~4M params
        "vocab_size": 8000,
        "min_frequency": 2,
        "max_seq_len": 256,
        "spm_vocab_size": 400,    # SentencePiece de juguete sobre el mini-corpus
        "n_bpe_merges": 200,      # merges del BPE didáctico (Sección 2)
    },
    "large": {           # Colab Pro (A100) — vocab del modelo de ~85M params
        "vocab_size": 32000,
        "min_frequency": 2,
        "max_seq_len": 1024,
        "spm_vocab_size": 600,
        "n_bpe_merges": 400,
    },
}

if PROFILE == "auto":
    PROFILE = "large" if VRAM_GB >= 30 else "small"

CONFIG = CONFIGS[PROFILE]
print("PROFILE = {}".format(PROFILE))
for k, v in CONFIG.items():
    print("  {:<15} = {}".format(k, v))

## El mini-corpus de trabajo

Para que el notebook sea **autocontenido** (corre en Colab sin clonar el repo ni descargar datasets), definimos inline un **mini-corpus técnico bilingüe** (es/en) que imita en miniatura el corpus real de la Fase 1: párrafos de los **21 dominios** del lab (`configs/corpus.yaml`), incluyendo código Python y SQL como texto.

Cada documento es un dict `{"domain", "lang", "text"}`. Este corpus se usa en **todas** las secciones: entrenaremos BPE, WordPiece, Unigram, SentencePiece y nuestro tokenizador final sobre él.

> **Advertencia de escala:** un corpus real de tokenización tiene de cientos de MB a decenas de GB. Con ~45 párrafos los vocabularios *saturan* muy por debajo de los 8k/32k objetivo — lo señalaremos cuando ocurra. La mecánica de los algoritmos, sin embargo, es idéntica a cualquier escala.

In [ ]:
# ── Mini-corpus técnico bilingüe: 21 dominios, ~45 documentos ────────
MINI_CORPUS = [
    # ── Machine Learning ──
    {"domain": "machine_learning", "lang": "es", "text": "El aprendizaje supervisado ajusta los parámetros de un modelo minimizando una función de pérdida sobre datos etiquetados. El sobreajuste aparece cuando el modelo memoriza el conjunto de entrenamiento y la regularización L2 penaliza pesos grandes para mitigarlo. La validación cruzada estima el error de generalización dividiendo el dataset en k particiones."},
    {"domain": "machine_learning", "lang": "en", "text": "A machine learning pipeline includes feature engineering, model selection and hyperparameter tuning. Gradient boosting ensembles like XGBoost combine weak decision trees, while the bias-variance tradeoff explains why more capacity is not always better generalization."},
    # ── Deep Learning ──
    {"domain": "deep_learning", "lang": "es", "text": "Una red neuronal profunda compone transformaciones afines con no linealidades como ReLU. La retropropagación calcula el gradiente de la pérdida respecto a cada peso aplicando la regla de la cadena, y el descenso de gradiente estocástico actualiza los parámetros por minibatches. El batch normalization estabiliza las activaciones intermedias."},
    {"domain": "deep_learning", "lang": "en", "text": "The transformer architecture replaced recurrence with self-attention, enabling parallel training over long sequences. Each attention head computes scaled dot products between queries and keys, and residual connections plus layer normalization keep gradients stable in deep stacks."},
    # ── Inteligencia Artificial ──
    {"domain": "inteligencia_artificial", "lang": "es", "text": "Un modelo de lenguaje autorregresivo predice el siguiente token condicionado al contexto previo. Los LLM se preentrenan con next-token prediction sobre corpus masivos y luego se alinean con instrucciones mediante fine-tuning supervisado y aprendizaje por refuerzo con retroalimentación humana."},
    {"domain": "inteligencia_artificial", "lang": "en", "text": "Artificial intelligence systems range from symbolic planners to large neural language models. Emergent capabilities such as in-context learning appear when scaling parameters, data and compute together, as described by neural scaling laws."},
    # ── Ingeniería Eléctrica ──
    {"domain": "ingenieria_electrica", "lang": "es", "text": "El transformador de potencia de la subestación eleva la tensión de 13.8 kV a 115 kV para reducir pérdidas por efecto Joule en la línea de transmisión. El aislamiento del devanado se verifica con pruebas de rigidez dieléctrica, y el interruptor de potencia despeja la falla en menos de cinco ciclos."},
    {"domain": "ingenieria_electrica", "lang": "es", "text": "La corriente de cortocircuito trifásico se calcula en p.u. sobre una base de 100 MVA. La impedancia equivalente de Thévenin vista desde la barra de la subestación determina la capacidad de interrupción requerida, y el relé de sobrecorriente se coordina con curvas de tiempo inverso."},
    {"domain": "ingenieria_electrica", "lang": "en", "text": "Power factor correction with capacitor banks reduces reactive power drawn from the grid. The protection relay measures phasors with a sampling rate synchronized by GPS, and circuit breakers interrupt fault currents of several kiloamperes."},
    # ── Sistemas Eléctricos de Potencia ──
    {"domain": "sistemas_potencia", "lang": "es", "text": "El flujo de carga resuelve las ecuaciones nodales de la red mediante el método de Newton-Raphson, obteniendo tensiones en p.u. y ángulos en cada barra. El despacho económico minimiza el costo de generación sujeto a límites de potencia activa y reactiva, y la pérdida de una línea se evalúa con análisis de contingencias N-1."},
    {"domain": "sistemas_potencia", "lang": "es", "text": "La demanda de energía eléctrica en Colombia presenta estacionalidad diaria y semanal. El pronóstico de demanda con redes neuronales alimenta el despacho del mercado mayorista, y la generación hidroeléctrica del sistema interconectado nacional cubre la punta de la curva de carga."},
    {"domain": "sistemas_potencia", "lang": "en", "text": "Frequency regulation keeps the grid at 60 Hz by balancing generation and load. Primary reserve responds in seconds through governor action, while automatic generation control corrects the area control error and tie-line flows between control areas."},
    # ── Ciencia de Datos ──
    {"domain": "ciencia_datos", "lang": "es", "text": "El análisis exploratorio de datos comienza con estadísticas descriptivas y visualizaciones: histogramas, diagramas de caja y matrices de correlación. La imputación de valores faltantes y la detección de atípicos preceden a cualquier modelo, porque basura que entra es basura que sale."},
    {"domain": "ciencia_datos", "lang": "en", "text": "A data scientist iterates between hypothesis, experiment and measurement. Feature importance from tree ensembles guides feature selection, and an A/B test with proper randomization quantifies the causal effect of a product change."},
    # ── Matemáticas ──
    {"domain": "matematicas", "lang": "es", "text": "El álgebra lineal sostiene todo el deep learning: una matriz define una transformación lineal, sus valores propios describen direcciones invariantes y la descomposición en valores singulares factoriza cualquier matriz como rotación, escalado y rotación. La norma de Frobenius mide el tamaño de una matriz."},
    {"domain": "matematicas", "lang": "en", "text": "The chain rule states that the derivative of a composition is the product of the derivatives. In several variables the Jacobian matrix generalizes this product, and Taylor expansion approximates a smooth function locally by a polynomial."},
    # ── Estadística ──
    {"domain": "estadistica", "lang": "es", "text": "La inferencia bayesiana actualiza la distribución a priori con la verosimilitud de los datos para obtener la posterior. El intervalo de confianza del 95 por ciento no es la probabilidad de que el parámetro caiga en el intervalo, error frecuente, sino la cobertura del procedimiento bajo repetición."},
    {"domain": "estadistica", "lang": "en", "text": "The maximum likelihood estimator chooses parameters that make the observed data most probable. Under regularity conditions it is asymptotically normal, and the central limit theorem explains why averages of many independent variables look Gaussian."},
    # ── Optimización ──
    {"domain": "optimizacion", "lang": "es", "text": "El gradiente descendente avanza en la dirección de máximo decrecimiento de la función objetivo con un paso controlado por la tasa de aprendizaje. Con momentum las oscilaciones se amortiguan, y Adam adapta el paso por coordenada usando medias móviles del gradiente y de su cuadrado."},
    {"domain": "optimizacion", "lang": "en", "text": "Convex optimization guarantees that any local minimum is global. The Lagrangian incorporates constraints with multipliers, the KKT conditions characterize optimality, and stochastic gradient descent trades exact gradients for cheap noisy estimates."},
    # ── Programación ──
    {"domain": "programacion", "lang": "es", "text": "Un buen programa separa responsabilidades: estructuras de datos claras, funciones puras donde sea posible y manejo explícito de errores. La complejidad algorítmica se expresa en notación O grande, y una tabla hash ofrece búsqueda en tiempo promedio constante."},
    {"domain": "programacion", "lang": "en", "text": "Version control with git enables branching, code review and reproducible releases. Unit tests document the expected behavior of each function, and continuous integration runs the test suite on every commit before merging."},
    # ── Python ──
    {"domain": "python", "lang": "es", "text": "En Python las list comprehensions expresan transformaciones de colecciones en una línea. Los generadores producen valores bajo demanda con yield, ahorrando memoria, y los context managers con with garantizan la liberación de recursos como archivos y conexiones."},
    {"domain": "python", "lang": "en", "text": "def train_epoch(model, loader, optimizer):\n    total_loss = 0.0\n    for batch in loader:\n        optimizer.zero_grad()\n        loss = model(batch).loss\n        loss.backward()\n        optimizer.step()\n        total_loss += loss.item()\n    return total_loss / len(loader)"},
    {"domain": "python", "lang": "en", "text": "import numpy as np\n\ndef softmax(x, axis=-1):\n    # estabilidad numerica: restar el maximo\n    z = x - np.max(x, axis=axis, keepdims=True)\n    e = np.exp(z)\n    return e / np.sum(e, axis=axis, keepdims=True)"},
    # ── PostgreSQL ──
    {"domain": "postgresql", "lang": "es", "text": "PostgreSQL implementa control de concurrencia multiversión: los lectores no bloquean a los escritores. Un índice B-tree acelera búsquedas por igualdad y rango, EXPLAIN ANALYZE muestra el plan de ejecución real y VACUUM recupera el espacio de las tuplas muertas."},
    {"domain": "postgresql", "lang": "en", "text": "SELECT d.domain_name, COUNT(*) AS docs, AVG(LENGTH(t.body)) AS avg_chars\nFROM documents t\nJOIN domains d ON d.id = t.domain_id\nWHERE t.lang IN ('es', 'en')\nGROUP BY d.domain_name\nHAVING COUNT(*) > 10\nORDER BY docs DESC;"},
    # ── MongoDB ──
    {"domain": "mongodb", "lang": "es", "text": "MongoDB almacena documentos BSON con esquema flexible dentro de colecciones. El pipeline de agregación encadena etapas como match, group y project, y un índice compuesto debe seguir la regla ESR: igualdad, sort, rango."},
    {"domain": "mongodb", "lang": "en", "text": "db.measurements.aggregate([\n  { $match: { substation: 'SUB-115', voltage_pu: { $gt: 0.95 } } },\n  { $group: { _id: '$feeder', avg_load_mw: { $avg: '$load_mw' } } },\n  { $sort: { avg_load_mw: -1 } }\n])"},
    # ── AWS ──
    {"domain": "aws", "lang": "es", "text": "En AWS una función Lambda escala automáticamente y se factura por milisegundo de ejecución. S3 ofrece almacenamiento de objetos con once nueves de durabilidad, DynamoDB entrega lecturas de un dígito de milisegundo y CloudWatch centraliza métricas y logs."},
    {"domain": "aws", "lang": "en", "text": "An event-driven architecture on AWS connects S3 uploads to Lambda through EventBridge rules. IAM roles grant least-privilege permissions, SQS buffers spikes between producers and consumers, and API Gateway exposes REST endpoints with throttling."},
    # ── Serverless Framework ──
    {"domain": "serverless", "lang": "es", "text": "El Serverless Framework describe funciones, eventos y recursos en un serverless.yml declarativo. Un comando de despliegue empaqueta el código, crea el stack de CloudFormation y publica cada función Lambda con sus variables de entorno y permisos IAM."},
    {"domain": "serverless", "lang": "en", "text": "service: demand-forecast\nprovider:\n  name: aws\n  runtime: python3.11\n  region: us-east-1\nfunctions:\n  ingest:\n    handler: handlers/ingest.run\n    events:\n      - schedule: rate(1 hour)"},
    # ── Infraestructura como Código ──
    {"domain": "iac", "lang": "es", "text": "La infraestructura como código versiona la red, el cómputo y los permisos igual que el código de aplicación. Terraform mantiene un estado declarativo y calcula el plan de cambios, mientras que la deriva de configuración se detecta comparando el estado real contra el deseado."},
    {"domain": "iac", "lang": "en", "text": "Immutable infrastructure replaces servers instead of patching them. A Terraform module encapsulates a reusable VPC with subnets and route tables, and the plan-apply workflow turns every infrastructure change into a reviewable pull request."},
    # ── APIs ──
    {"domain": "apis", "lang": "es", "text": "Una API REST modela recursos con sustantivos y verbos HTTP: GET para leer, POST para crear, PUT para reemplazar. Los códigos de estado comunican el resultado, la paginación con cursores evita respuestas gigantes y un contrato OpenAPI documenta cada endpoint."},
    {"domain": "apis", "lang": "en", "text": "Idempotency keys make retries safe in payment APIs. Rate limiting with token buckets protects the backend, OAuth2 issues scoped access tokens, and semantic versioning signals breaking changes to API consumers."},
    # ── Arquitectura de Software ──
    {"domain": "arquitectura_software", "lang": "es", "text": "La arquitectura hexagonal separa el dominio de los adaptadores de entrada y salida: la lógica de negocio no conoce la base de datos ni el framework web. Los eventos de dominio desacoplan módulos y el patrón CQRS divide los modelos de lectura y escritura."},
    {"domain": "arquitectura_software", "lang": "en", "text": "Microservices trade a monolith for independently deployable services with their own data stores. The price is distributed transactions, eventual consistency and observability overhead, so start with a modular monolith unless scale demands otherwise."},
    # ── DevOps ──
    {"domain": "devops", "lang": "es", "text": "Un pipeline CI/CD compila, prueba y despliega en cada push. Los despliegues azul-verde mantienen dos entornos idénticos para cambiar el tráfico sin caída, los canarios exponen la nueva versión a un porcentaje de usuarios y el rollback automático se activa al violar un SLO."},
    {"domain": "devops", "lang": "en", "text": "Observability rests on three pillars: metrics, logs and traces. Prometheus scrapes time series, Grafana dashboards reveal saturation, and an error budget tells the team when to stop shipping features and invest in reliability."},
    # ── Data Engineering ──
    {"domain": "data_engineering", "lang": "es", "text": "Un pipeline ETL extrae datos de las fuentes, los transforma con reglas de limpieza y deduplicación, y los carga en el warehouse. La orquestación con Airflow define dependencias como un grafo acíclico dirigido, y el particionado por fecha acelera las consultas analíticas."},
    {"domain": "data_engineering", "lang": "en", "text": "Batch and streaming converge in modern lakehouse architectures. Schema evolution must be backward compatible, idempotent writes make reprocessing safe, and data quality checks halt the pipeline before corrupt records reach downstream consumers."},
    # ── Sistemas Distribuidos ──
    {"domain": "sistemas_distribuidos", "lang": "es", "text": "El teorema CAP obliga a elegir entre consistencia y disponibilidad ante una partición de red. Los relojes lógicos de Lamport ordenan eventos sin reloj global, y el consenso con Raft elige un líder que replica el log en la mayoría de los nodos."},
    {"domain": "sistemas_distribuidos", "lang": "en", "text": "Exactly-once delivery is impossible without cooperation from the receiver: deduplication plus idempotent handlers approximate it. Consistent hashing spreads keys across nodes with minimal reshuffling when the cluster grows or shrinks."},
]

print("Documentos: {}".format(len(MINI_CORPUS)))
print("Dominios distintos: {}".format(len(set(d["domain"] for d in MINI_CORPUS))))
print("Idiomas: {}".format(sorted(set(d["lang"] for d in MINI_CORPUS))))

In [ ]:
# ── Exploración rápida del mini-corpus ───────────────────────────────
corpus_df = pd.DataFrame(MINI_CORPUS)
corpus_df["n_chars"] = corpus_df["text"].str.len()
corpus_df["n_words"] = corpus_df["text"].str.split().str.len()

ALL_TEXTS = [d["text"] for d in MINI_CORPUS]           # lista de strings
FULL_TEXT = "\n".join(ALL_TEXTS)                        # todo el corpus junto

print("Total: {:,} caracteres | {:,} palabras (split por espacios)".format(
    corpus_df["n_chars"].sum(), corpus_df["n_words"].sum()))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
by_domain = corpus_df.groupby("domain")["n_chars"].sum().sort_values()
by_domain.plot.barh(ax=axes[0], color="steelblue")
axes[0].set_title("Caracteres por dominio (mini-corpus)")
axes[0].set_xlabel("caracteres")

corpus_df.groupby("lang")["n_chars"].sum().plot.bar(
    ax=axes[1], color=["indianred", "seagreen"], rot=0)
axes[1].set_title("Caracteres por idioma")
plt.tight_layout()
plt.show()

corpus_df.head(6)

<a id="sec1"></a>
## 1. ¿Por qué tokenizar? El problema del vocabulario abierto

Un modelo de lenguaje define una distribución sobre secuencias de **símbolos discretos** $x_1, \dots, x_T$ de un vocabulario finito $\mathcal{V}$:

$$P(x_1, \dots, x_T) = \prod_{t=1}^{T} P(x_t \mid x_{<t}), \qquad x_t \in \mathcal{V}, \quad |\mathcal{V}| = V$$

La capa de salida del transformer es un softmax sobre $V$ clases y la capa de entrada una tabla de embeddings $W_E \in \mathbb{R}^{V \times d_{model}}$. **El texto crudo no es discreto ni finito**: hay que convertirlo en una secuencia de ids — eso es tokenizar. La elección del esquema fija dos costos que compiten:

- **Costo en vocabulario** ($V$): la tabla de embeddings y el softmax pesan $O(V \cdot d_{model})$. Con `tie_embeddings: true` (como en `configs/small.yaml`) se paga una sola vez, pero sigue siendo el bloque más grande de un modelo pequeño.
- **Costo en longitud** ($T$): la atención cuesta $O(T^2)$ y el contexto es fijo (`max_seq_len` = 256 en SMALL). Cada token gastado en deletrear una palabra es contexto perdido.

### Las tres opciones

**Caracteres** — $\mathcal{V}$ = alfabeto (~100–1000 símbolos). Nunca hay palabras fuera de vocabulario (OOV), pero $T$ explota: "subestación" son 11 tokens y el modelo debe *aprender* la morfología desde cero gastando capas en ello.

**Palabras** — $\mathcal{V}$ = palabras vistas en el corpus. $T$ mínimo, pero el vocabulario del lenguaje natural es **abierto**: la ley de Heaps dice que el número de palabras distintas crece como $V(n) \approx K n^{\beta}$ con $\beta \approx 0.4$–$0.6$ — *nunca deja de crecer* con el corpus $n$. Todo lo no visto colapsa al token `<UNK>`: "QLoRA", "p.u.", "EventBridge" se vuelven indistinguibles. Además español e inglés multiplican formas: *transformador, transformadores, transformer, transformers* serían 4 entradas independientes sin compartir nada.

**Subpalabras** — el punto medio que usan todos los LLM modernos: palabras frecuentes quedan enteras (1 token), palabras raras se descomponen en piezas reutilizables (`sub + estación`). $V$ acotado por diseño (8k–256k), OOV imposible (en el límite se cae a caracteres/bytes) y la morfología queda *expuesta* en la segmentación.

### Tabla de trade-offs

| Criterio | Caracteres | Palabras | Subpalabras |
|---|---|---|---|
| Tamaño de vocabulario $V$ | ~10²–10³ | 10⁵–10⁷ (y creciendo) | **fijado por diseño: 8k–256k** |
| Longitud de secuencia $T$ | muy larga (≈ chars) | mínima (≈ palabras) | intermedia (~1.2–2 tokens/palabra) |
| Palabras OOV | imposibles | **inevitables** (`<UNK>`) | imposibles (fallback a piezas) |
| Morfología compartida | implícita (el modelo la aprende) | nula | **explícita** (sub-, -ción, -ing) |
| Costo softmax/embeddings | mínimo | enorme | moderado |
| Costo de atención $O(T^2)$ | enorme | mínimo | moderado |
| Multilingüe / código | trivial | pésimo | bueno (si se entrena con esos datos) |

**Conclusión adelantada:** los LLM modernos usan subpalabras porque convierten un problema *abierto* (¿qué palabras existen?) en uno *cerrado* (¿cuáles son las $V$ piezas que mejor comprimen mi corpus?). Las secciones 2–6 derivan los cuatro algoritmos que responden esa pregunta.

In [ ]:
# ── Implementación: los tres esquemas sobre el mismo texto ───────────
import re

def tokenize_chars(text):
    """Tokenización por caracteres: cada char es un token."""
    return list(text)

def tokenize_words(text):
    """Tokenización por palabras: secuencias alfanuméricas o signos sueltos."""
    return re.findall(r"\w+|[^\w\s]", text, re.UNICODE)

def tokenize_subwords_naive(text, max_len=4):
    """Subpalabras 'de juguete': trocea cada palabra en pedazos de <= max_len chars.
    (Solo para ilustrar el punto medio; BPE hara esto BIEN en la Seccion 2.)"""
    pieces = []
    for w in tokenize_words(text):
        pieces.extend(w[i:i + max_len] for i in range(0, len(w), max_len))
    return pieces

sample = ("La subestación eleva la tensión a 115 kV y el transformador "
          "de potencia opera a 0.98 p.u. con gradiente descendente no tiene nada que ver.")

for name, fn in [("caracteres", tokenize_chars),
                 ("palabras", tokenize_words),
                 ("subpalabras (naive)", tokenize_subwords_naive)]:
    toks = fn(sample)
    print("{:<22} T = {:>3} | primeros 12: {}".format(name, len(toks), toks[:12]))

In [ ]:
# ── Visualización: longitud de secuencia por esquema, todo el corpus ─
schemes = {
    "caracteres": tokenize_chars,
    "palabras": tokenize_words,
    "subpalabras (naive)": tokenize_subwords_naive,
}
lengths = {name: [len(fn(t)) for t in ALL_TEXTS] for name, fn in schemes.items()}
len_df = pd.DataFrame({
    "esquema": np.repeat(list(lengths.keys()), len(ALL_TEXTS)),
    "T (tokens por documento)": np.concatenate(list(lengths.values())),
})

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
totals = {k: sum(v) for k, v in lengths.items()}
axes[0].bar(totals.keys(), totals.values(),
            color=["indianred", "seagreen", "steelblue"])
axes[0].set_title("Tokens totales del mini-corpus por esquema")
axes[0].set_ylabel("tokens")
for i, (k, v) in enumerate(totals.items()):
    axes[0].text(i, v, "{:,}".format(v), ha="center", va="bottom")

sns.boxplot(data=len_df, x="esquema", y="T (tokens por documento)", ax=axes[1],
            hue="esquema", palette=["indianred", "seagreen", "steelblue"], legend=False)
axes[1].set_title("Distribución de T por documento")
plt.tight_layout()
plt.show()

ratio = totals["caracteres"] / totals["palabras"]
print("Caracteres genera {:.1f}x más tokens que palabras: con max_seq_len = {} ".format(
    ratio, CONFIG["max_seq_len"]))
print("el contexto efectivo se divide por ese mismo factor.")

<a id="sec2"></a>
## 2. BPE (Byte-Pair Encoding) desde cero

BPE nace como algoritmo de **compresión** (Gage, 1994) y Sennrich et al. (2016) lo adaptan a traducción automática. La idea es un algoritmo **greedy de construcción de vocabulario**:

> Empieza con el vocabulario mínimo (caracteres). Repite $k$ veces: encuentra el **par de símbolos adyacentes más frecuente** en el corpus y fusiónalo en un símbolo nuevo.

### Derivación: por qué "el par más frecuente"

Sea $c(a,b)$ el número de veces que el par $(a, b)$ aparece adyacente en el corpus tokenizado con el vocabulario actual. Al fusionar $(a,b) \to ab$, la longitud total del corpus en tokens cae exactamente en $c(a,b)$ (cada ocurrencia pasa de 2 símbolos a 1):

$$T_{nuevo} = T_{actual} - c(a, b)$$

Elegir el par de máxima frecuencia es, por tanto, el **paso greedy que maximiza la compresión inmediata** del corpus, pagando una sola entrada nueva de vocabulario:

$$(a^*, b^*) = \arg\max_{(a,b)} \; c(a,b)$$

No es óptimo globalmente (es greedy), pero es simple, determinista y funciona notablemente bien. Tras $k$ merges: $|\mathcal{V}| = |\text{alfabeto}| + k$. El **vocab objetivo se controla con $k$** — esa es la perilla `vocab_size` de cualquier tokenizador BPE.

### Detalles del algoritmo clásico

1. **Pre-segmentación**: el corpus se parte en palabras y se trabaja con un diccionario `{palabra: frecuencia}` — los merges *no cruzan* fronteras de palabra (eso lo cambiará SentencePiece, §5).
2. **Marcador de fin de palabra** `</w>`: distingue `s` al final de palabra (plural) de `s` interna, y permite reconstruir el texto al decodificar.
3. **Codificación** de texto nuevo: se aplican los merges aprendidos **en el mismo orden** en que se aprendieron (el orden ES el modelo).

### Ejemplo a mano

Mini-corpus de 3 palabras (con frecuencias): `sol`×5, `sola`×3, `solar`×2. Representación inicial por caracteres + `</w>`:

```
s o l </w>      (×5)
s o l a </w>    (×3)
s o l a r </w>  (×2)
```

**Conteo de pares** (frecuencia = suma sobre palabras que lo contienen):

| Par | Conteo | Cálculo |
|---|---|---|
| `(s, o)` | **10** | 5 + 3 + 2 |
| `(o, l)` | **10** | 5 + 3 + 2 |
| `(l, </w>)` | 5 | solo en "sol" |
| `(l, a)` | 5 | 3 + 2 |
| `(a, </w>)` | 3 | solo en "sola" |
| `(a, r)` | 2 | solo en "solar" |
| `(r, </w>)` | 2 | solo en "solar" |

**Merge 1**: empate entre `(s,o)` y `(o,l)` con 10; tomamos el primero → nuevo símbolo `so`. El corpus pasa a `so l </w>`, `so l a </w>`, `so l a r </w>`.

**Merge 2**: ahora `(so, l)` = 10 es el máximo → símbolo `sol`. Corpus: `sol </w>`, `sol a </w>`, `sol a r </w>`.

**Merge 3**: `(sol, </w>)` = 5 y `(sol, a)` = 5 empatan; tomamos `(sol, </w>)` → la palabra completa `sol</w>` ya es **un solo token**. Las raíces compartidas se fusionan primero (frecuencia acumulada) y los sufijos después: BPE descubre morfología sin saber qué es un morfema.

Verifica a mano: tras el merge 3, $T = 5 \cdot 1 + 3 \cdot 3 + 2 \cdot 4 = 22$ tokens, contra $T = 5 \cdot 4 + 3 \cdot 5 + 2 \cdot 6 = 47$ inicial. Compresión de 2.1x con solo 3 merges.

In [ ]:
# ── Implementación de BPE en Python puro ─────────────────────────────
from collections import Counter, defaultdict

END = "</w>"   # marcador de fin de palabra

def get_word_freqs(texts):
    """Corpus -> dict {palabra_como_tupla_de_simbolos: frecuencia}.
    Cada palabra arranca segmentada en caracteres + END."""
    freqs = Counter()
    for text in texts:
        for w in re.findall(r"\w+|[^\w\s]", text.lower(), re.UNICODE):
            freqs[tuple(w) + (END,)] += 1
    return dict(freqs)

def get_pair_stats(word_freqs):
    """Cuenta cada par de simbolos ADYACENTES, ponderado por frecuencia de palabra."""
    stats = defaultdict(int)
    for word, freq in word_freqs.items():
        for i in range(len(word) - 1):
            stats[(word[i], word[i + 1])] += freq
    return stats

def merge_pair(pair, word_freqs):
    """Aplica un merge: reemplaza cada ocurrencia adyacente de `pair` por su fusion."""
    merged = pair[0] + pair[1]            # el simbolo nuevo es la concatenacion
    new_freqs = {}
    for word, freq in word_freqs.items():
        new_word, i = [], 0
        while i < len(word):
            if i < len(word) - 1 and (word[i], word[i + 1]) == pair:
                new_word.append(merged)   # fusiona el par
                i += 2
            else:
                new_word.append(word[i])
                i += 1
        new_freqs[tuple(new_word)] = freq
    return new_freqs

def train_bpe(texts, num_merges):
    """Entrena BPE: devuelve (lista de merges en orden, historial, estado final)."""
    word_freqs = get_word_freqs(texts)
    merges, history = [], []
    for step in range(num_merges):
        stats = get_pair_stats(word_freqs)
        if not stats:
            break                          # ya no quedan pares (corpus saturado)
        best = max(stats, key=stats.get)   # par mas frecuente (greedy)
        merges.append(best)
        history.append({"merge": step + 1, "par": best,
                        "nuevo_token": best[0] + best[1], "frecuencia": stats[best]})
        word_freqs = merge_pair(best, word_freqs)
    return merges, history, word_freqs

def bpe_encode_word(word, merges):
    """Codifica una palabra nueva aplicando los merges EN SU ORDEN de aprendizaje."""
    symbols = list(word.lower()) + [END]
    for pair in merges:                    # el orden de los merges ES el modelo
        i, out = 0, []
        while i < len(symbols):
            if i < len(symbols) - 1 and (symbols[i], symbols[i + 1]) == pair:
                out.append(symbols[i] + symbols[i + 1])
                i += 2
            else:
                out.append(symbols[i])
                i += 1
        symbols = out
    return symbols

def bpe_encode(text, merges):
    """Codifica un texto: palabra por palabra (los merges no cruzan palabras)."""
    tokens = []
    for w in re.findall(r"\w+|[^\w\s]", text, re.UNICODE):
        tokens.extend(bpe_encode_word(w, merges))
    return tokens

# Sanity check con el ejemplo a mano de la celda anterior
toy_merges, toy_hist, _ = train_bpe(["sol " * 5 + "sola " * 3 + "solar " * 2], 3)
print("Merges del ejemplo a mano:", toy_merges)
assert toy_merges[0] == ("s", "o") and toy_merges[1] == ("so", "l")
assert bpe_encode_word("sol", toy_merges) == ["sol</w>"]
print("OK: coincide con la derivación manual.")

In [ ]:
# ── Entrenar sobre el mini-corpus: primeros 30 merges paso a paso ────
N_MERGES = CONFIG["n_bpe_merges"]
bpe_merges, bpe_history, bpe_final_state = train_bpe(ALL_TEXTS, N_MERGES)

merges_df = pd.DataFrame(bpe_history[:30])
merges_df["par"] = merges_df["par"].apply(lambda p: "({} + {})".format(p[0], p[1]))
print("Entrenados {} merges (pedidos {}).".format(len(bpe_merges), N_MERGES))
print("\nPrimeros 30 merges — observa: primero pares de chars frecuentes")
print("(es/en comparten 'de', 'en', 'ti'...), luego sufijos y palabras enteras:\n")
merges_df.style.hide(axis="index")

In [ ]:
# ── ¿Cómo se segmenta "transformador" a medida que crecen los merges? ─
word = "transformador"
checkpoints = [0, 5, 10, 25, 50, 100, 150, len(bpe_merges)]
checkpoints = sorted(set(min(c, len(bpe_merges)) for c in checkpoints))

rows = []
for k in checkpoints:
    seg = bpe_encode_word(word, bpe_merges[:k])
    rows.append({"merges": k, "n_tokens": len(seg), "segmentación": " | ".join(seg)})
seg_df = pd.DataFrame(rows)
print(seg_df.to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 5))
ax.step(seg_df["merges"], seg_df["n_tokens"], where="post",
        marker="o", color="darkorange", lw=2)
for _, r in seg_df.iterrows():
    ax.annotate(r["segmentación"].replace(" | ", "·"),
                (r["merges"], r["n_tokens"]),
                textcoords="offset points", xytext=(6, 8), fontsize=8, rotation=20)
ax.set_xlabel("número de merges aprendidos")
ax.set_ylabel("tokens para '{}'".format(word))
ax.set_title("La palabra técnica se 'compacta' al crecer el vocabulario BPE")
plt.tight_layout()
plt.show()

In [ ]:
# ── Compresión del corpus completo en función de los merges ──────────
def corpus_token_count(texts, merges):
    return sum(len(bpe_encode(t, merges)) for t in texts)

ks = sorted(set(list(range(0, len(bpe_merges) + 1, max(1, len(bpe_merges) // 12)))
                + [len(bpe_merges)]))
counts = [corpus_token_count(ALL_TEXTS, bpe_merges[:k]) for k in ks]

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(ks, counts, marker="o", color="steelblue")
ax.set_xlabel("número de merges k  (|V| = alfabeto + k)")
ax.set_ylabel("tokens totales del corpus T")
ax.set_title("Cada merge reduce T en c(a,b): rendimientos decrecientes")
ax.axhline(counts[0], ls=":", color="gray")
ax.text(ks[-1] * 0.55, counts[0], "nivel caracteres (k=0)", va="bottom", fontsize=9)
plt.tight_layout()
plt.show()

print("k=0 (caracteres): T = {:,}".format(counts[0]))
print("k={}: T = {:,}  ->  compresión {:.2f}x".format(
    ks[-1], counts[-1], counts[0] / counts[-1]))
print("\nEjemplo de codificación:")
print(bpe_encode("la subestación eleva la tensión", bpe_merges))

### Observaciones sobre lo que aprendió BPE

- Los **primeros merges** son bigramas de caracteres ultra-frecuentes compartidos por español e inglés (`de`, `en`, `er`, `on`...): la frecuencia manda, no el idioma.
- Luego emergen **sufijos morfológicos** (`ción</w>`, `ing</w>`, `mente</w>`) y **palabras funcionales completas** (`la</w>`, `the</w>`, `de</w>`).
- La curva de compresión tiene **rendimientos decrecientes**: el primer merge ahorra cientos de tokens, el merge 200 ahorra un puñado. Por eso los vocabularios reales paran en 32k–256k: más allá, cada entrada nueva del softmax cuesta más de lo que comprime.
- Limitación pendiente: si el texto trae un carácter **nunca visto** (un emoji, un ideograma), este BPE no tiene símbolo para él → `<UNK>`. Lo resuelve la siguiente sección.

<a id="sec3"></a>
## 3. Byte-level BPE: el truco de GPT-2

El BPE clásico arranca de **caracteres Unicode**. Problema: Unicode tiene ~150 000 caracteres asignados; o incluyes todos (vocabulario base gigante, casi todo basura para tu corpus) o aceptas `<UNK>` para lo no visto. GPT-2 (Radford et al., 2019) corta el nudo:

> Opera sobre **bytes UTF-8**, no sobre caracteres. El alfabeto base son los **256 bytes posibles** — *cualquier* texto del universo es una secuencia de esos 256 símbolos. **`<UNK>` deja de existir por construcción.**

El costo: un carácter no-ASCII ocupa varios bytes (`ñ` → 2 bytes, `⚡` → 3, un emoji → 4). En el peor caso (texto en un alfabeto no visto en entrenamiento) la secuencia se alarga ~4x, pero **nunca** pierde información. BPE luego aprende merges sobre bytes igual que antes, y las secuencias frecuentes de bytes (= palabras frecuentes) terminan siendo un token.

### La tabla bytes → unicode

Detalle de ingeniería: los archivos de vocabulario de GPT-2 son texto (JSON), y muchos bytes son de control o invisibles (`0x00`, `\n`, espacio...). GPT-2 define una **biyección** $\beta: \{0,...,255\} \to$ caracteres Unicode *imprimibles*:

- Los 188 bytes "visibles y seguros" (rangos `!`–`~`, `¡`–`¬`, `®`–`ÿ`) se mapean **a sí mismos**.
- Los 68 restantes (controles, espacio, DEL...) se desplazan a `256 + n` (caracteres `Ā`, `ā`, `Ă`...).

Por eso en el vocab de GPT-2 ves tokens como `Ġhello`: `Ġ` es el byte `0x20` (espacio) mapeado a 256+32. **El espacio inicial forma parte del token** — primer paso hacia "no pre-segmentar" (§5).

In [ ]:
# ── La biyección bytes -> unicode de GPT-2, implementada ─────────────
def bytes_to_unicode():
    """Mapa byte (0-255) -> caracter unicode imprimible. Identico a GPT-2."""
    # bytes "seguros": se representan a si mismos
    bs = (list(range(ord("!"), ord("~") + 1))      # ASCII imprimible
          + list(range(ord("¡"), ord("¬") + 1))    # latin-1 imprimible (parte 1)
          + list(range(ord("®"), ord("ÿ") + 1)))   # latin-1 imprimible (parte 2)
    cs = bs[:]
    n = 0
    for b in range(256):                            # bytes problematicos:
        if b not in bs:                             # controles, espacio, DEL...
            bs.append(b)
            cs.append(256 + n)                      # se desplazan a 256+n
            n += 1
    return dict(zip(bs, [chr(c) for c in cs]))

BYTE_ENCODER = bytes_to_unicode()
BYTE_DECODER = {v: k for k, v in BYTE_ENCODER.items()}
assert len(BYTE_ENCODER) == 256 and len(BYTE_DECODER) == 256  # biyeccion completa

# Tabla de muestra del mapeo
sample_bytes = [0, 9, 10, 32, 33, 65, 97, 126, 127, 160, 161, 195, 241, 255]
map_df = pd.DataFrame({
    "byte (dec)": sample_bytes,
    "byte (hex)": ["0x{:02X}".format(b) for b in sample_bytes],
    "significado": ["NUL", "TAB", "\\n", "espacio", "!", "A", "a", "~", "DEL",
                    "NBSP", "¡", "0xC3 (lider UTF-8 de ñ/á...)", "ñ en latin-1", "ÿ"],
    "se mapea a": [BYTE_ENCODER[b] for b in sample_bytes],
})
remapped = sum(1 for b in range(256) if ord(BYTE_ENCODER[b]) >= 256)
print("Bytes remapeados a la zona 256+: {} | identidad: {}".format(remapped, 256 - remapped))
map_df

In [ ]:
# ── Demostración: jamás UNK, a costa de secuencias más largas ────────
def to_byte_symbols(text):
    """Texto -> simbolos byte-level (el punto de partida del BPE de GPT-2)."""
    return [BYTE_ENCODER[b] for b in text.encode("utf-8")]

def from_byte_symbols(symbols):
    """Inversa exacta: simbolos -> bytes -> texto."""
    return bytes(BYTE_DECODER[s] for s in symbols).decode("utf-8")

tests = ["subestación", "p.u.", "señal ⚡ 115 kV", "naïve façade", "東京", "🤖🔌"]
rows = []
for t in tests:
    syms = to_byte_symbols(t)
    assert from_byte_symbols(syms) == t        # round-trip SIN PERDIDA, siempre
    rows.append({"texto": t, "chars": len(t), "bytes (símbolos base)": len(syms),
                 "símbolos": "".join(syms)})
demo_df = pd.DataFrame(rows)
print("Round-trip byte-level verificado para los {} casos (incluye emoji y CJK).".format(len(tests)))
print("Un tokenizador por caracteres entrenado en nuestro corpus habría dado <UNK>")
print("en ⚡, 東, 京 y los emojis. Byte-level NUNCA necesita <UNK>.\n")
demo_df

<a id="sec4"></a>
## 4. WordPiece: el criterio de verosimilitud de BERT

WordPiece (Schuster & Nakajima, 2012; usado por BERT) es estructuralmente idéntico a BPE — vocabulario inicial de caracteres, merges iterativos — pero cambia **qué par se fusiona**. En lugar de frecuencia cruda, maximiza la ganancia de **verosimilitud** del corpus bajo un modelo unigrama de tokens.

### Derivación del score

Bajo un modelo unigrama, la log-verosimilitud del corpus segmentado es $\mathcal{L} = \sum_{i} \log P(t_i)$ con $P(t) = \frac{c(t)}{N}$ (frecuencia relativa del token, $N$ tokens totales). Al fusionar $(a, b) \to ab$, las $c(a,b)$ ocurrencias del par dejan de aportar $\log P(a) + \log P(b)$ y pasan a aportar $\log P(ab)$. El cambio de verosimilitud por ocurrencia es:

$$\Delta \mathcal{L} \propto \log P(ab) - \log P(a) - \log P(b) = \log \frac{P(ab)}{P(a)\,P(b)}$$

Sustituyendo $P(\cdot) = c(\cdot)/N$ y como $c(ab) = c(a,b)$ (cada ocurrencia adyacente del par se vuelve el token nuevo), maximizar $\Delta\mathcal{L}$ equivale a maximizar:

$$\text{score}(a, b) = \frac{c(a, b)}{c(a) \cdot c(b)}$$

que es exactamente el **PMI (información mutua puntual)** del par, sin la constante $N$. La lectura probabilística: fusiona los símbolos que aparecen juntos **más de lo que predice el azar** ($P(ab) \gg P(a)P(b)$), no los que simplemente aparecen mucho.

### BPE vs WordPiece en un ejemplo a mano

Con los conteos del ejemplo de §2 (`sol`×5, `sola`×3, `solar`×2): $c(s)=c(o)=c(l)=10$, $c(a)=5$, $c(r)=2$.

| Par | $c(a,b)$ | BPE (frecuencia) | WordPiece: $\frac{c(a,b)}{c(a)c(b)}$ |
|---|---|---|---|
| `(s, o)` | 10 | **gana (10)** | $10/(10 \cdot 10) = 0.10$ |
| `(l, a)` | 5 | 5 | $5/(10 \cdot 5) = 0.10$ |
| `(a, r)` | 2 | 2 | $2/(5 \cdot 2) = \mathbf{0.20}$ **gana** |

BPE fusiona el par más *frecuente* `(s,o)`; WordPiece prefiere `(a,r)`: aunque raro, **cada vez que aparece `a`, ¿qué tan probable es que le siga `r`?** — el par es más "exclusivo". WordPiece tiende a capturar antes unidades con fuerte asociación interna (morfemas raros pero cohesivos).

### El prefijo `##`

BERT marca los tokens **internos** de palabra con `##`: `tokenización → token + ##iza + ##ción`. Es la convención inversa a `</w>` (marca continuación en vez de fin) y permite detokenizar sin ambigüedad: un token sin `##` inicia palabra nueva. Costo: `ción` al inicio de palabra y `##ción` interna son **dos entradas distintas** del vocabulario.

In [ ]:
# ── WordPiece simplificado: mismo esqueleto que BPE, otro criterio ───
def get_word_freqs_wp(texts):
    """Como en BPE, pero con la convencion ##: primer char libre, resto con ##."""
    freqs = Counter()
    for text in texts:
        for w in re.findall(r"\w+", text.lower(), re.UNICODE):
            pieces = (w[0],) + tuple("##" + ch for ch in w[1:])
            freqs[pieces] += 1
    return dict(freqs)

def wp_merge_symbol(a, b):
    """Fusion respetando ##: (token, ##iza) -> token##iza... sin el ## interno."""
    return a + b[2:] if b.startswith("##") else a + b

def train_wordpiece(texts, num_merges):
    """WordPiece: en cada paso fusiona el par con MAXIMO score = c(par)/(c(a)*c(b))."""
    word_freqs = get_word_freqs_wp(texts)
    merges, history = [], []
    for step in range(num_merges):
        pair_counts = defaultdict(int)
        unit_counts = defaultdict(int)
        for word, freq in word_freqs.items():
            for sym in word:
                unit_counts[sym] += freq
            for i in range(len(word) - 1):
                pair_counts[(word[i], word[i + 1])] += freq
        if not pair_counts:
            break
        # score(a,b) = c(a,b) / (c(a) * c(b))  — el criterio de verosimilitud
        best = max(pair_counts,
                   key=lambda p: pair_counts[p] / (unit_counts[p[0]] * unit_counts[p[1]]))
        score = pair_counts[best] / (unit_counts[best[0]] * unit_counts[best[1]])
        merged = wp_merge_symbol(*best)
        merges.append(best)
        history.append({"merge": step + 1, "par": "({} + {})".format(*best),
                        "nuevo_token": merged, "freq_par": pair_counts[best],
                        "score": round(score, 5)})
        # aplicar el merge (reutiliza la mecanica de BPE con la fusion ##)
        new_freqs = {}
        for word, freq in word_freqs.items():
            out, i = [], 0
            while i < len(word):
                if i < len(word) - 1 and (word[i], word[i + 1]) == best:
                    out.append(merged)
                    i += 2
                else:
                    out.append(word[i])
                    i += 1
            new_freqs[tuple(out)] = freq
        word_freqs = new_freqs
    return merges, history, word_freqs

wp_merges, wp_history, wp_state = train_wordpiece(ALL_TEXTS, 120)
print("Primeros 20 merges de WordPiece (observa los scores, no las frecuencias):")
pd.DataFrame(wp_history[:20]).style.hide(axis="index")

In [ ]:
# ── Comparar QUÉ aprende cada criterio + segmentación con ## ─────────
def wp_encode_word(word, merges):
    """Codifica con los merges WordPiece en orden (greedy, como BPE)."""
    w = word.lower()
    symbols = [w[0]] + ["##" + ch for ch in w[1:]]
    for pair in merges:
        out, i = [], 0
        while i < len(symbols):
            if i < len(symbols) - 1 and (symbols[i], symbols[i + 1]) == pair:
                out.append(wp_merge_symbol(*pair))
                i += 2
            else:
                out.append(symbols[i])
                i += 1
        symbols = out
    return symbols

for w in ["tokenización", "subestación", "transformador", "gradiente"]:
    print("{:<15} BPE: {:<42} WP: {}".format(
        w,
        " ".join(bpe_encode_word(w, bpe_merges[:120])),
        " ".join(wp_encode_word(w, wp_merges))))

# ¿Cuanto difieren los vocabularios resultantes? (tokens nuevos creados)
bpe_new = set(h["nuevo_token"].replace(END, "") for h in bpe_history[:120])
wp_new = set(h["nuevo_token"].replace("##", "") for h in wp_history[:120])
inter = bpe_new & wp_new
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(["solo BPE", "comunes", "solo WordPiece"],
       [len(bpe_new - wp_new), len(inter), len(wp_new - bpe_new)],
       color=["steelblue", "gray", "indianred"])
ax.set_title("Tokens creados en 120 merges: frecuencia (BPE) vs PMI (WordPiece)")
ax.set_ylabel("número de tokens nuevos")
plt.tight_layout()
plt.show()
print("Coincidencia: {:.0f}% — mismo esqueleto, criterio distinto, vocabularios distintos.".format(
    100 * len(inter) / max(1, len(bpe_new | wp_new))))

<a id="sec5"></a>
## 5. SentencePiece: tokenizar sin pre-segmentar

Todo lo anterior asume un paso previo: **partir el texto en palabras** por espacios. Esa suposición es frágil:

- **Idiomas sin espacios**: chino, japonés, tailandés no separan palabras con espacios.
- **Código fuente**: en Python la indentación *es semántica* (`    return x` ≠ `return x`); en SQL los saltos de línea y espacios múltiples importan para el round-trip exacto. Pre-segmentar por espacios **destruye** esa información: `decode(encode(x)) ≠ x`.
- **Idiomas aglutinantes** y URLs, fórmulas, `p.u.`: la noción de "palabra" es borrosa.

SentencePiece (Kudo & Richardson, 2018) elimina el paso: trata el texto como una **secuencia cruda de caracteres donde el espacio es un carácter más**, visible como `▁` (U+2581, *lower one eighth block*):

$$\text{"el flujo de carga"} \;\to\; \text{▁el ▁flujo ▁de ▁carga}$$

Sobre esa secuencia entrena BPE **o** Unigram (§6) sin fronteras impuestas. Consecuencias:

1. **Round-trip exacto**: `decode(encode(x)) == x` siempre (lossless), crítico para código.
2. Tokens **multi-palabra** posibles (`▁de▁la`) y tokens que capturan indentación (`▁▁▁▁`).
3. Un solo modelo `.model` autocontenido con normalización incluida (NFKC) — sin reglas de pre-tokenización dependientes del idioma.

> Lo usan T5, Llama 1/2, Mistral, Gemma. La diferencia con el byte-level de GPT-2 (§3) es el *alfabeto base* (caracteres con cobertura configurable + fallback a bytes, vs bytes puros); la filosofía "el espacio es parte del token" es compartida.

In [ ]:
# ── Entrenar un SentencePiece pequeño sobre el mini-corpus ───────────
import sentencepiece as spm
import tempfile

SPM_OK = False
try:
    # SentencePiece entrena desde archivo: volcamos el mini-corpus a un txt
    spm_dir = tempfile.mkdtemp(prefix="spm_lab_")
    corpus_txt = os.path.join(spm_dir, "mini_corpus.txt")
    with open(corpus_txt, "w", encoding="utf-8") as f:
        for t in ALL_TEXTS:
            f.write(t.replace("\n", " ") + "\n")    # un documento por linea

    spm_prefix = os.path.join(spm_dir, "spm_unigram")
    vocab_try = CONFIG["spm_vocab_size"]
    while vocab_try >= 100:
        try:
            spm.SentencePieceTrainer.train(
                input=corpus_txt,
                model_prefix=spm_prefix,
                vocab_size=vocab_try,            # pequeño: el corpus es diminuto
                model_type="unigram",            # el algoritmo de la Sección 6
                character_coverage=1.0,          # es/en: cubrir todos los chars
                normalization_rule_name="nfkc",  # normalización Unicode integrada
                pad_id=3,
            )
            SPM_OK = True
            break
        except RuntimeError as e:                 # vocab_size > piezas posibles
            print("vocab_size={} falló ({}), probando menos...".format(
                vocab_try, str(e)[:60]))
            vocab_try -= 100
    if SPM_OK:
        sp = spm.SentencePieceProcessor(model_file=spm_prefix + ".model")
        print("SentencePiece (unigram) entrenado: vocab = {}".format(sp.vocab_size()))
except Exception as e:
    print("No se pudo entrenar SentencePiece:", e)
    print("El resto del notebook no depende de esta sección.")

In [ ]:
# ── Inspección: el espacio como ▁ y el round-trip exacto ─────────────
if SPM_OK:
    examples = [
        "El flujo de carga resuelve las ecuaciones nodales de la red.",
        "the transformer architecture replaced recurrence",
        "def softmax(x):\n    return np.exp(x) / np.exp(x).sum()",
        "SELECT * FROM documents WHERE lang = 'es';",
    ]
    for ex in examples:
        pieces = sp.encode(ex, out_type=str)
        decoded = sp.decode(sp.encode(ex))
        print("texto   :", repr(ex[:60]))
        print("piezas  :", pieces[:18])
        print("roundtrip exacto:", decoded == ex)
        print("-" * 70)

    # muestra del vocabulario: piezas con ▁ (inician 'palabra') vs internas
    vocab_pieces = [sp.id_to_piece(i) for i in range(sp.vocab_size())]
    with_marker = [p for p in vocab_pieces if p.startswith("▁")]
    print("Piezas totales: {} | con ▁ (inician tras espacio): {} ({:.0f}%)".format(
        len(vocab_pieces), len(with_marker),
        100 * len(with_marker) / len(vocab_pieces)))
    print("Ejemplos con ▁ :", with_marker[10:24])
    print("Ejemplos internos:", [p for p in vocab_pieces if not p.startswith("▁")][10:24])
else:
    print("SentencePiece no disponible; sección omitida.")

**Nota sobre el round-trip con código:** observa que el ejemplo con `def softmax` recupera el texto **exactamente**, incluida la indentación de 4 espacios. Un tokenizador con pre-segmentación por espacios (BPE clásico, WordPiece de BERT) habría colapsado esos espacios. Para un LLM que debe *generar* Python válido, esto no es un detalle: es la diferencia entre código que corre y código con `IndentationError`.

<a id="sec6"></a>
## 6. Unigram LM: tokenización probabilística

BPE y WordPiece son **constructivos**: empiezan de caracteres y *agregan* tokens con merges, y segmentan de forma determinista. Unigram (Kudo, 2018) invierte todo: parte de un vocabulario **grande** (todas las subcadenas frecuentes) y lo **poda**, tratando la segmentación como variable latente de un modelo probabilístico.

### El modelo

Cada pieza $t \in \mathcal{V}$ tiene probabilidad $p(t)$, con $\sum_{t} p(t) = 1$. Una segmentación $\mathbf{s} = (t_1, \dots, t_M)$ de la secuencia $X$ tiene probabilidad (independencia unigrama):

$$P(\mathbf{s}) = \prod_{i=1}^{M} p(t_i)$$

Como muchas segmentaciones producen el mismo texto, la verosimilitud marginal de $X$ suma sobre todas las segmentaciones válidas $\mathcal{S}(X)$:

$$P(X) = \sum_{\mathbf{s} \in \mathcal{S}(X)} P(\mathbf{s}) \qquad\Rightarrow\qquad \mathcal{L} = \sum_{X \in corpus} \log P(X)$$

### Los tres ingredientes

**(1) Viterbi** — para *segmentar*, buscamos $\mathbf{s}^* = \arg\max_{\mathbf{s}} P(\mathbf{s})$. Programación dinámica sobre las posiciones del texto: si $\delta(j)$ es la mejor log-probabilidad de segmentar los primeros $j$ caracteres,

$$\delta(j) = \max_{i < j,\; X_{i:j} \in \mathcal{V}} \left[ \delta(i) + \log p(X_{i:j}) \right], \qquad \delta(0) = 0$$

Costo $O(n \cdot \ell_{max})$ con $\ell_{max}$ = longitud máxima de pieza. Igual que el Viterbi de HMMs, pero sobre posiciones de corte.

**(2) EM** — las $p(t)$ se estiman maximizando $\mathcal{L}$ con segmentación latente: el paso E calcula las ocurrencias esperadas de cada pieza bajo las probabilidades actuales (en la práctica con forward-backward; aquí usaremos **hard-EM**: la segmentación Viterbi como si fuera la verdadera), y el paso M re-estima $p(t) = \frac{\text{count}(t)}{\sum_{t'} \text{count}(t')}$.

**(3) Poda** — se eliminan iterativamente las piezas cuya remoción **menos daña** $\mathcal{L}$ (manteniendo siempre los caracteres base para garantizar cobertura), hasta llegar al `vocab_size` objetivo. Esto da el control *exacto* del tamaño que BPE logra con $k$ merges.

### Ejemplo a mano: Viterbi sobre "casa"

Vocabulario: $p(\text{c})=0.05$, $p(\text{a})=0.10$, $p(\text{s})=0.05$, $p(\text{ca})=0.04$, $p(\text{sa})=0.03$, $p(\text{casa})=0.01$. Tres segmentaciones candidatas:

| Segmentación | $P(\mathbf{s})$ | $\log P$ |
|---|---|---|
| `c·a·s·a` | $0.05 \cdot 0.1 \cdot 0.05 \cdot 0.1 = 2.5 \times 10^{-5}$ | $-10.60$ |
| `ca·sa` | $0.04 \cdot 0.03 = 1.2 \times 10^{-3}$ | $-6.73$ |
| `casa` | $0.01$ | $-4.61$ |

Gana `casa` entera: aunque $p(\text{casa})$ es la más baja *por pieza*, usar **menos piezas** evita multiplicar probabilidades pequeñas. El modelo equilibra automáticamente piezas largas-raras contra cortas-frecuentes. (Bonus del enfoque probabilístico: muestrear segmentaciones alternativas da *subword regularization*, un aumento de datos gratuito.)

In [ ]:
# ── Viterbi para segmentar con un vocabulario probabilístico ─────────
import math

def viterbi_segment(text, logp, max_piece_len=12):
    """Mejor segmentacion de `text` bajo log-probs `logp` (dict pieza -> logP).
    dp[j] = (mejor logprob de text[:j], indice i del corte previo)."""
    n = len(text)
    dp = [(-math.inf, -1)] * (n + 1)
    dp[0] = (0.0, -1)
    for j in range(1, n + 1):
        for i in range(max(0, j - max_piece_len), j):
            piece = text[i:j]
            if piece in logp and dp[i][0] > -math.inf:
                cand = dp[i][0] + logp[piece]       # delta(i) + log p(X_ij)
                if cand > dp[j][0]:
                    dp[j] = (cand, i)
    if dp[n][0] == -math.inf:
        return None, -math.inf                       # sin cobertura
    pieces, j = [], n                                # backtracking de cortes
    while j > 0:
        i = dp[j][1]
        pieces.append(text[i:j])
        j = i
    return pieces[::-1], dp[n][0]

# Verificación contra el ejemplo a mano
toy_p = {"c": 0.05, "a": 0.10, "s": 0.05, "ca": 0.04, "sa": 0.03, "casa": 0.01}
toy_logp = {k: math.log(v) for k, v in toy_p.items()}
seg, lp = viterbi_segment("casa", toy_logp)
print("Viterbi('casa') = {} con logP = {:.2f}".format(seg, lp))
assert seg == ["casa"] and abs(lp - math.log(0.01)) < 1e-9
print("OK: coincide con el cálculo manual (-4.61).")

In [ ]:
# ── Entrenamiento Unigram simplificado: init grande + hard-EM + poda ─
def init_unigram_vocab(texts, max_piece_len=8, max_pieces=2500):
    """Vocabulario semilla: todas las subcadenas frecuentes (dentro de palabras),
    siempre incluyendo los caracteres sueltos (cobertura garantizada)."""
    counts = Counter()
    words = Counter()
    for t in texts:
        for w in re.findall(r"\w+", t.lower(), re.UNICODE):
            words[w] += 1
    for w, f in words.items():
        for i in range(len(w)):
            for j in range(i + 1, min(i + max_piece_len, len(w)) + 1):
                counts[w[i:j]] += f
    chars = {p for p in counts if len(p) == 1}
    top = sorted((p for p in counts if len(p) > 1),
                 key=lambda p: -counts[p] * len(p))[: max_pieces - len(chars)]
    vocab = chars | set(top)
    return {p: counts[p] for p in vocab}, words

def train_unigram(texts, target_size=600, n_em_iters=4, prune_frac=0.2):
    """Hard-EM + poda. Devuelve (logp final, historial de verosimilitud)."""
    counts, words = init_unigram_vocab(texts)
    total = sum(counts.values())
    logp = {p: math.log(c / total) for p, c in counts.items()}
    history = []
    while True:
        for it in range(n_em_iters):
            # E (hard): segmentar cada palabra con Viterbi y contar piezas
            new_counts, corpus_ll = Counter(), 0.0
            for w, f in words.items():
                seg, lp = viterbi_segment(w, logp)
                corpus_ll += f * lp
                for piece in seg:
                    new_counts[piece] += f
            # M: re-estimar p(t) con suavizado leve para piezas no usadas
            total = sum(new_counts.values())
            logp = {p: math.log((new_counts.get(p, 0) + 1e-6) / total) for p in logp}
            history.append({"vocab": len(logp), "logL": corpus_ll})
        if len(logp) <= target_size:
            break
        # Poda: quitar el prune_frac de piezas multi-char menos usadas/probables
        multi = sorted((p for p in logp if len(p) > 1), key=lambda p: logp[p])
        n_prune = min(len(multi), max(1, int(prune_frac * len(logp))),
                      len(logp) - target_size)
        for p in multi[:n_prune]:
            del logp[p]                      # los chars sueltos jamas se podan
    return logp, pd.DataFrame(history)

uni_logp, uni_hist = train_unigram(ALL_TEXTS, target_size=600)
print("Vocabulario Unigram final: {} piezas".format(len(uni_logp)))
top_pieces = sorted(uni_logp, key=uni_logp.get, reverse=True)
print("Piezas más probables:", [p for p in top_pieces if len(p) > 1][:18])

In [ ]:
# ── Visualizar el EM y comparar segmentaciones ───────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(uni_hist.index, uni_hist["logL"], marker=".", color="purple")
axes[0].set_xlabel("iteración EM (con podas intercaladas)")
axes[0].set_ylabel("log-verosimilitud del corpus")
axes[0].set_title("Hard-EM: la verosimilitud sube; cada poda la golpea un poco")
ax2 = axes[0].twinx()
ax2.plot(uni_hist.index, uni_hist["vocab"], color="gray", ls="--", alpha=0.6)
ax2.set_ylabel("|V| (gris, punteado)")

words_test = ["transformador", "subestación", "tokenización", "retropropagación", "gradiente"]
rows = []
for w in words_test:
    seg_u, lp_u = viterbi_segment(w, uni_logp)
    rows.append({"palabra": w,
                 "Unigram (Viterbi)": " · ".join(seg_u),
                 "logP": round(lp_u, 1),
                 "BPE (greedy, §2)": " · ".join(
                     s.replace(END, "") for s in bpe_encode_word(w, bpe_merges))})
cmp_df = pd.DataFrame(rows)
axes[1].axis("off")
tbl = axes[1].table(cellText=cmp_df.values, colLabels=cmp_df.columns,
                    loc="center", cellLoc="left")
tbl.auto_set_font_size(False)
tbl.set_fontsize(8)
tbl.scale(1, 1.6)
axes[1].set_title("Unigram (global, probabilístico) vs BPE (greedy)")
plt.tight_layout()
plt.show()

**Lo esencial de Unigram:** la segmentación no es una secuencia fija de merges sino la solución de un problema de **inferencia** ($\arg\max$ sobre segmentaciones). Por eso puede (a) encontrar segmentaciones globalmente mejores que el greedy de BPE, (b) asignar probabilidad a segmentaciones alternativas (*subword regularization*: muestrear en vez de Viterbi durante el entrenamiento del LLM mejora la robustez), y (c) controlar el vocab de forma exacta vía poda. Es el algoritmo por defecto de SentencePiece — el `model_type="unigram"` que entrenamos en §5 ejecutó exactamente esta receta (con EM completo en vez de hard-EM y una inicialización más sofisticada con el algoritmo de Trie + ESA).

<a id="sec7"></a>
## 7. Comparación de tokenizadores reales: GPT-2, BERT y Llama

Ya conocemos las cuatro familias; veámoslas en producción:

| Tokenizador | Algoritmo | Vocab | Marca |
|---|---|---|---|
| GPT-2 | BPE byte-level (§2+§3) | 50 257 | `Ġ` = espacio |
| BERT multilingüe | WordPiece (§4) | 119 547 | `##` = continuación |
| Llama-2 | SentencePiece-BPE (§5) | 32 000 | `▁` = espacio |

### La métrica clave: fertilidad

$$\text{fertilidad} = \frac{\#\text{tokens}}{\#\text{palabras}}$$

Una fertilidad de 1.0 significa "cada palabra es un token" (ideal); 2.5 significa que el modelo gasta 2.5 posiciones de contexto por palabra. La fertilidad depende de **qué corpus vio el tokenizador al entrenarse**: GPT-2 se entrenó con texto mayormente en inglés ⇒ fertilidad baja en inglés, alta en español. Para un LLM bilingüe técnico como el nuestro, esto define cuánto contexto efectivo tiene cada idioma — y es el argumento central para **entrenar tokenizador propio** (§8).

In [ ]:
# ── Cargar tokenizadores reales con fallbacks robustos ───────────────
from transformers import AutoTokenizer

def try_load(candidates):
    """Intenta cargar el primer tokenizador disponible de la lista."""
    for name in candidates:
        try:
            tok = AutoTokenizer.from_pretrained(name)
            print("  OK  {} (vocab = {:,})".format(name, tok.vocab_size))
            return tok, name
        except Exception as e:
            print("  --  {} no disponible ({})".format(name, type(e).__name__))
    return None, None

real_toks = {}
print("GPT-2 (BPE byte-level):")
tok, _ = try_load(["gpt2", "openai-community/gpt2"])
if tok:
    real_toks["GPT-2"] = tok
print("BERT (WordPiece):")
tok, _ = try_load(["bert-base-multilingual-cased", "bert-base-uncased"])
if tok:
    real_toks["BERT"] = tok
print("Tipo Llama (SentencePiece BPE):")
tok, _ = try_load(["hf-internal-testing/llama-tokenizer",
                   "NousResearch/Llama-2-7b-hf", "openlm-research/open_llama_3b"])
if tok:
    real_toks["Llama"] = tok

if not real_toks:
    print("\nSin conexión a HF Hub: las celdas siguientes se degradan con aviso.")
else:
    print("\nTokenizadores cargados:", list(real_toks.keys()))

In [ ]:
# ── El mismo texto técnico por tres tokenizadores ────────────────────
probe_texts = {
    "técnico ES": "La subestación eleva la tensión a 115 kV y el transformador de potencia opera a 0.98 p.u.",
    "técnico EN": "The substation steps up the voltage to 115 kV and the power transformer operates at 0.98 p.u.",
    "código Python": "def softmax(x):\n    z = x - np.max(x)\n    return np.exp(z) / np.exp(z).sum()",
}

if real_toks:
    for label, text in probe_texts.items():
        print("=" * 72)
        print("[{}]  {}".format(label, repr(text[:70])))
        n_words = len(text.split())
        for name, tok in real_toks.items():
            pieces = tok.tokenize(text)
            print("  {:<7} {:>3} tokens (fert. {:.2f}): {}".format(
                name, len(pieces), len(pieces) / n_words,
                pieces[:11] + ["..."] if len(pieces) > 11 else pieces))
else:
    print("Sección omitida: no hay tokenizadores de HF disponibles.")

In [ ]:
# ── Fertilidad por idioma y por dominio: el sesgo del corpus ─────────
if real_toks:
    rows = []
    for doc in MINI_CORPUS:
        n_words = len(doc["text"].split())
        for name, tok in real_toks.items():
            n_tokens = len(tok.tokenize(doc["text"]))
            rows.append({"dominio": doc["domain"], "idioma": doc["lang"],
                         "tokenizador": name, "fertilidad": n_tokens / n_words})
    fert_df = pd.DataFrame(rows)

    by_lang = fert_df.pivot_table(index="tokenizador", columns="idioma",
                                  values="fertilidad", aggfunc="mean").round(2)
    by_lang["penalización ES vs EN"] = (by_lang["es"] / by_lang["en"]).round(2)
    print("Fertilidad media (tokens/palabra) por idioma:")
    print(by_lang.to_string())
    print("\nGPT-2 'cobra' más caro el español: misma frase, más tokens,")
    print("menos contexto efectivo para el modelo.")
else:
    print("Sección omitida: no hay tokenizadores de HF disponibles.")

In [ ]:
# ── Heatmap dominio × tokenizador (seaborn) ──────────────────────────
if real_toks:
    pivot = fert_df.pivot_table(index="dominio", columns="tokenizador",
                                values="fertilidad", aggfunc="mean")
    pivot = pivot.loc[pivot.mean(axis=1).sort_values(ascending=False).index]

    fig, ax = plt.subplots(figsize=(9, 9))
    sns.heatmap(pivot, annot=True, fmt=".2f", cmap="RdYlGn_r",
                cbar_kws={"label": "fertilidad (tokens/palabra)"}, ax=ax)
    ax.set_title("Fertilidad por dominio y tokenizador\n(rojo = caro: más tokens por palabra)")
    plt.tight_layout()
    plt.show()

    worst = pivot.mean(axis=1).idxmax()
    best = pivot.mean(axis=1).idxmin()
    print("Dominio más 'caro': {} | más 'barato': {}".format(worst, best))
    print("Los dominios en español y con jerga técnica (subestación, p.u.) fragmentan")
    print("más; los de código/inglés (visto masivamente por GPT-2) fragmentan menos.")
else:
    print("Sección omitida: no hay tokenizadores de HF disponibles.")

<a id="sec8"></a>
## 8. Entrenar nuestro propio tokenizador con `tokenizers`

Nuestra implementación de §2 es didáctica pero es $O(V \cdot N)$ por re-escaneo completo; la librería [`tokenizers`](https://github.com/huggingface/tokenizers) (Rust) entrena el mismo algoritmo en segundos sobre GB de texto usando colas de prioridad y actualización incremental de conteos. Siguiendo el contrato del lab — *desde cero primero, librería después* — ahora construimos el tokenizador **oficial de la Etapa 1**, pieza a pieza:

| Componente | Elección | Por qué |
|---|---|---|
| `normalizer` | **NFKC** | Unifica representaciones Unicode: `ñ` compuesta vs descompuesta (`n` + combinante), ligaduras (`ﬁ`→`fi`), anchos completos. Sin esto, la "misma" palabra genera entradas duplicadas de vocab. |
| `pre_tokenizer` | **ByteLevel** | El truco de GPT-2 (§3): alfabeto base de 256 bytes, cero `<UNK>`, espacios dentro del token (`Ġ`). |
| `model` | **BPE** | Merges por frecuencia (§2). |
| `decoder` | **ByteLevel** | Invierte el mapeo bytes→unicode para el round-trip exacto. |
| `vocab_size` | **8 000 (SMALL) / 32 000 (LARGE)** | Espeja `configs/small.yaml` / `large.yaml`: debe coincidir con el `vocab_size` del modelo (¡la tabla de embeddings se dimensiona con esto!). |

> Con el mini-corpus el vocab **saturará** muy por debajo de 8k (no hay suficientes pares que fusionar). En la Fase 1 real, este mismo código corre sobre el corpus completo de 21 dominios y sí alcanza el objetivo.

In [ ]:
# ── Construcción y entrenamiento del tokenizador del lab ─────────────
from tokenizers import Tokenizer, models, normalizers, pre_tokenizers, decoders, trainers

own_tok = Tokenizer(models.BPE())                       # modelo: BPE (§2)
own_tok.normalizer = normalizers.NFKC()                 # normalización Unicode
own_tok.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)  # §3
own_tok.decoder = decoders.ByteLevel()

trainer = trainers.BpeTrainer(
    vocab_size=CONFIG["vocab_size"],                    # 8k SMALL / 32k LARGE
    min_frequency=CONFIG["min_frequency"],
    special_tokens=["<|endoftext|>", "<|pad|>"],        # separador de documentos
    initial_alphabet=pre_tokenizers.ByteLevel.alphabet(),  # los 256 bytes SIEMPRE
    show_progress=False,
)

own_tok.train_from_iterator(ALL_TEXTS, trainer)
actual_vocab = own_tok.get_vocab_size()
print("vocab_size pedido: {:,} | aprendido: {:,}".format(
    CONFIG["vocab_size"], actual_vocab))
if actual_vocab < CONFIG["vocab_size"]:
    print("-> Saturó: el mini-corpus no da para más merges (esperado; ver §8).")

In [ ]:
# ── Guardar + round-trip verificado con asserts ──────────────────────
import unicodedata

# Carpeta de salida: tokenizer/trained/ del repo, o /content en Colab
if IN_COLAB:
    TOK_DIR = "/content/tokenizer/trained"
elif os.path.isdir(os.path.join("..", "tokenizer")):
    TOK_DIR = os.path.join("..", "tokenizer", "trained")    # corriendo desde notebooks/
else:
    TOK_DIR = os.path.join("tokenizer", "trained")
os.makedirs(TOK_DIR, exist_ok=True)

TOK_PATH = os.path.join(TOK_DIR, "tokenizer_nb02_{}.json".format(PROFILE))
own_tok.save(TOK_PATH)
print("Guardado en:", os.path.abspath(TOK_PATH))

# Round-trip: decode(encode(x)) debe devolver x (tras NFKC, que es idempotente)
roundtrip_cases = ALL_TEXTS + [
    "subestación de 115 kV a 0.98 p.u. ⚡",
    "def f(x):\n    return x ** 2   # comentario",
    "SELECT lang, COUNT(*) FROM docs GROUP BY lang;",
]
reloaded = Tokenizer.from_file(TOK_PATH)                 # verificar el archivo
for text in roundtrip_cases:
    expected = unicodedata.normalize("NFKC", text)       # NFKC es parte del contrato
    for t in (own_tok, reloaded):
        enc = t.encode(text)
        assert t.decode(enc.ids) == expected, "round-trip falló: " + repr(text[:50])
print("Round-trip encode->decode verificado en {} textos (incluye código y emoji),".format(
    len(roundtrip_cases)))
print("tanto con el tokenizador en memoria como con el archivo recargado.")

In [ ]:
# ── El tokenizador propio en acción ──────────────────────────────────
for label, text in probe_texts.items():
    enc = own_tok.encode(text)
    n_words = len(text.split())
    print("[{}] {} tokens (fertilidad {:.2f})".format(label, len(enc.ids),
                                                      len(enc.ids) / n_words))
    print("   ", enc.tokens[:14])
print("\nNota: 'Ġ' marca el espacio inicial (§3); el vocabulario aprendió")
print("términos del corpus como tokens completos pese a su rareza en la web.")

<a id="sec9"></a>
## 9. Anatomía del vocabulario aprendido

Un tokenizador entrenado es un artefacto inspeccionable. Preguntas de auditoría que haríamos a cualquier vocab antes de gastar cómputo entrenando un LLM con él:

1. **¿Longitud media de token?** Más largo = más compresión (correlaciona con chars/token de §11).
2. **¿Composición?** % de tokens alfabéticos, numéricos, de símbolos, con espacio inicial. Un vocab lleno de basura numérica (`00`, `001`, `0012`...) desperdicia embeddings.
3. **¿Los términos de nuestro dominio quedaron enteros?** Si "subestación" cuesta 5 tokens, el modelo aprenderá electricidad con desventaja estructural.

Para inspeccionar tokens byte-level hay que **deshacer el mapeo** de §3 (`Ġ` → espacio, etc.) — reutilizamos nuestro `BYTE_DECODER`.

In [ ]:
# ── Estadísticas del vocab: longitud y composición ───────────────────
def readable_token(tok_str):
    """Token byte-level -> texto legible (deshace el mapeo bytes->unicode de §3)."""
    try:
        return bytes(BYTE_DECODER[ch] for ch in tok_str).decode("utf-8", errors="replace")
    except KeyError:                       # tokens especiales (<|endoftext|>)
        return tok_str

vocab = own_tok.get_vocab()                            # dict token -> id
vocab_rows = []
for tok_str, tok_id in vocab.items():
    txt = readable_token(tok_str)
    stripped = txt.strip()
    if txt.startswith("<|"):
        cls = "especial"
    elif stripped.isalpha():
        cls = "alfabético"
    elif stripped.isdigit():
        cls = "numérico"
    elif stripped == "":
        cls = "espacios"
    elif any(c.isalnum() for c in stripped):
        cls = "mixto"
    else:
        cls = "símbolos"
    vocab_rows.append({"token": tok_str, "texto": txt, "len_chars": len(txt),
                       "clase": cls, "inicia_palabra": txt.startswith(" ")})
vocab_df = pd.DataFrame(vocab_rows)

print("Tokens: {:,} | longitud media: {:.2f} chars | mediana: {:.0f}".format(
    len(vocab_df), vocab_df["len_chars"].mean(), vocab_df["len_chars"].median()))
print("Con espacio inicial (Ġ): {:.0f}%".format(100 * vocab_df["inicia_palabra"].mean()))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
vocab_df["len_chars"].hist(bins=range(0, 22), ax=axes[0], color="steelblue",
                           edgecolor="white")
axes[0].set_title("Distribución de longitud de token (chars)")
axes[0].set_xlabel("caracteres por token")
comp = vocab_df["clase"].value_counts()
axes[1].bar(comp.index, 100 * comp.values / len(vocab_df),
            color=sns.color_palette("muted", len(comp)))
axes[1].set_title("Composición del vocabulario (%)")
axes[1].set_ylabel("% del vocab")
plt.tight_layout()
plt.show()

In [ ]:
# ── Tokens más largos y cobertura de términos del dominio ────────────
longest = vocab_df.sort_values("len_chars", ascending=False).head(15)
print("Los 15 tokens más largos (delatan qué memorizó el vocab):")
print(longest[["texto", "len_chars", "clase"]].to_string(index=False))

TECH_TERMS = [
    # eléctricos                      # ML / datos
    "subestación", "transformador",   "gradiente", "retropropagación",
    "cortocircuito", "interruptor",   "sobreajuste", "embedding",
    "tensión", "reactiva",            "regularización", "softmax",
    # infraestructura
    "serverless", "microservicios", "kubernetes", "pipeline",
]
print("\nCobertura: tokens que cuesta cada término (con espacio previo, caso real):")
for term in TECH_TERMS:
    n = len(own_tok.encode(" " + term).ids)
    seg = own_tok.encode(" " + term).tokens
    flag = "ENTERO" if n == 1 else "{} piezas".format(n)
    print("  {:<18} {:>9}   {}".format(term, flag, seg if n > 1 else ""))

<a id="sec10"></a>
## 10. Frecuencia de tokens y ley de Zipf

La **ley de Zipf** (1949) establece que en lenguaje natural la frecuencia de la $r$-ésima palabra más común decae como potencia del rango:

$$f(r) \propto \frac{1}{r^{s}}, \qquad s \approx 1 \;\Rightarrow\; \log f = C - s \log r$$

En escala log-log es una **recta de pendiente $-s$**. ¿Sobrevive Zipf a la tokenización subword? Aproximadamente sí, con desviaciones características: la cabeza se aplana (los tokens ultra-frecuentes son piezas funcionales compartidas) y la cola cae abrupto (la poda/saturación del vocab corta la cola infinita de palabras raras).

### Entropía de la distribución de tokens

La entropía de Shannon de la distribución unigrama de tokens:

$$H = -\sum_{t \in \mathcal{V}} p(t) \log_2 p(t) \quad \text{[bits/token]}, \qquad H \le \log_2 V \;\text{(uniforme)}$$

mide cuánta "sorpresa" media carga cada token. Dos lecturas para LLMs: (a) $H$ es una **cota inferior trivial** del cross-entropy loss que puede alcanzar un modelo unigrama — el LLM debe hacerlo mejor usando contexto; (b) la razón $H / \log_2 V$ (eficiencia) dice qué fracción de la "capacidad de código" del vocabulario se usa: una distribución Zipf siempre está lejos del máximo uniforme.

In [ ]:
# ── Rank vs frecuencia en log-log + ajuste de pendiente ──────────────
token_counts = Counter()
for t in ALL_TEXTS:
    token_counts.update(own_tok.encode(t).ids)

freqs = np.array(sorted(token_counts.values(), reverse=True), dtype=float)
ranks = np.arange(1, len(freqs) + 1, dtype=float)

# Ajuste de pendiente s en la zona central (la cabeza y la cola se desvían)
lo, hi = max(1, len(freqs) // 20), max(2, int(len(freqs) * 0.8))
slope, intercept = np.polyfit(np.log(ranks[lo:hi]), np.log(freqs[lo:hi]), 1)

fig, ax = plt.subplots(figsize=(9, 6))
ax.loglog(ranks, freqs, ".", ms=4, color="steelblue", label="tokens del corpus")
fit = np.exp(intercept) * ranks ** slope
ax.loglog(ranks[lo:hi], fit[lo:hi], "r-", lw=2,
          label="ajuste: s = {:.2f}".format(-slope))
ax.set_xlabel("rango del token (log)")
ax.set_ylabel("frecuencia (log)")
ax.set_title("Ley de Zipf sobre el corpus tokenizado (propio, {})".format(PROFILE))
ax.legend()
plt.tight_layout()
plt.show()

print("Pendiente estimada s = {:.2f} (Zipf clásico: s ≈ 1; corpus diminuto = ruido).".format(-slope))
print("Tokens distintos usados: {:,} de {:,} del vocab.".format(
    len(token_counts), own_tok.get_vocab_size()))

In [ ]:
# ── Entropía de la distribución de tokens ────────────────────────────
counts_arr = np.array(list(token_counts.values()), dtype=float)
p = counts_arr / counts_arr.sum()                 # distribución unigrama p(t)
H = float(-(p * np.log2(p)).sum())                # bits por token

V_total = own_tok.get_vocab_size()
V_usado = len(token_counts)
H_max_total = math.log2(V_total)                  # cota: uniforme sobre todo el vocab
H_max_usado = math.log2(V_usado)                  # cota: uniforme sobre lo usado

print("Entropía H               = {:.2f} bits/token".format(H))
print("log2(V usado = {:>5,})    = {:.2f} bits  -> eficiencia {:.0f}%".format(
    V_usado, H_max_usado, 100 * H / H_max_usado))
print("log2(V total = {:>5,})    = {:.2f} bits  -> eficiencia {:.0f}%".format(
    V_total, H_max_total, 100 * H / H_max_total))

fig, ax = plt.subplots(figsize=(8, 4))
bars = ["H real", "log2(V usado)", "log2(V total)"]
vals = [H, H_max_usado, H_max_total]
ax.barh(bars, vals, color=["purple", "lightgray", "gray"])
for i, v in enumerate(vals):
    ax.text(v, i, " {:.2f} bits".format(v), va="center")
ax.set_xlabel("bits/token")
ax.set_title("Zipf implica H muy por debajo del máximo uniforme")
plt.tight_layout()
plt.show()
print("\nLectura para el LLM: un modelo unigrama lograría loss = H·ln2 = {:.2f} nats;".format(H * math.log(2)))
print("todo lo que el transformer baje de ahí lo gana modelando el CONTEXTO.")

<a id="sec11"></a>
## 11. Compresión del lenguaje: chars/token y bytes/token

Un tokenizador es, literalmente, un **compresor**: convierte $B$ bytes de texto en $T$ tokens. Las métricas:

$$\text{chars/token} = \frac{\#\text{caracteres}}{\#\text{tokens}} \qquad \text{bytes/token} = \frac{\#\text{bytes UTF-8}}{\#\text{tokens}}$$

(difieren en es/en porque las tildes y la `ñ` ocupan 2 bytes). ¿Por qué importa para el LLM?

- **Contexto efectivo**: con `max_seq_len = 256` y 4.0 chars/token, una ventana "ve" ~1024 caracteres; con 2.5 chars/token, solo ~640. Misma arquitectura, 60% más de información por forward pass.
- **Costo de entrenamiento**: el corpus completo medido en tokens (= pasos × batch × seq_len necesarios para una época) se reduce proporcionalmente.
- **Equidad entre dominios e idiomas**: si el español comprime peor que el inglés, el modelo dedica menos "atención por carácter" al español — un sesgo silencioso que se decide *aquí*, antes de entrenar nada.

In [ ]:
# ── chars/token y bytes/token por dominio e idioma ───────────────────
comp_rows = []
for doc in MINI_CORPUS:
    n_tok = len(own_tok.encode(doc["text"]).ids)
    comp_rows.append({
        "dominio": doc["domain"], "idioma": doc["lang"],
        "chars/token": len(doc["text"]) / n_tok,
        "bytes/token": len(doc["text"].encode("utf-8")) / n_tok,
    })
comp_df = pd.DataFrame(comp_rows)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
dom_means = (comp_df.groupby("dominio")[["chars/token", "bytes/token"]]
             .mean().sort_values("chars/token"))
dom_means.plot.barh(ax=axes[0], color=["steelblue", "darkorange"])
axes[0].set_title("Compresión por dominio (tokenizador propio)")
axes[0].set_xlabel("ratio (más alto = mejor compresión)")

lang_means = comp_df.groupby("idioma")[["chars/token", "bytes/token"]].mean()
lang_means.plot.bar(ax=axes[1], color=["steelblue", "darkorange"], rot=0)
axes[1].set_title("Compresión por idioma")
for cont in axes[1].containers:
    axes[1].bar_label(cont, fmt="%.2f")
plt.tight_layout()
plt.show()

ct = comp_df["chars/token"].mean()
print("Media global: {:.2f} chars/token -> max_seq_len = {} tokens".format(
    ct, CONFIG["max_seq_len"]))
print("equivale a una ventana de ~{:,} caracteres de contexto efectivo.".format(
    int(ct * CONFIG["max_seq_len"])))

<a id="sec12"></a>
## 12. Cobertura técnica: tokenizador propio vs GPT-2

La prueba reina para nuestro caso de uso: ¿cuántos **términos técnicos clave** de los 21 dominios quedan como **1 token** y cuántos se fragmentan? Un término fragmentado no es ilegible para el modelo, pero:

- consume varias posiciones de contexto cada vez que aparece,
- obliga al modelo a *componer* su significado desde piezas (a veces sin relación morfológica real: `sub + est + ación`),
- y en generación multiplica los puntos donde un sampling desafortunado puede romper la palabra.

Comparamos nuestro tokenizador (entrenado sobre texto técnico es/en) contra GPT-2 (entrenado sobre web mayormente en inglés). Es una comparación *injusta a propósito*: ilustra exactamente por qué los modelos de dominio (BloombergGPT, Galactica, code models) entrenan tokenizador propio.

In [ ]:
# ── Términos clave: 1 token vs fragmentado, propio vs GPT-2 ──────────
KEY_TERMS = [
    "subestación", "transformador", "cortocircuito", "interruptor", "tensión",
    "reactiva", "aislamiento", "demanda", "gradiente", "descendente",
    "retropropagación", "sobreajuste", "regularización", "embedding", "softmax",
    "tokenización", "verosimilitud", "optimización", "serverless", "kubernetes",
    "microservicios", "pipeline", "warehouse", "consenso", "idempotente",
]

gpt2_tok = real_toks.get("GPT-2")
cov_rows = []
for term in KEY_TERMS:
    row = {"término": term}
    row["propio"] = len(own_tok.encode(" " + term).ids)      # espacio = caso real
    if gpt2_tok is not None:
        row["GPT-2"] = len(gpt2_tok.tokenize(" " + term))
    cov_rows.append(row)
cov_df = pd.DataFrame(cov_rows).set_index("término")

tok_cols = [c for c in cov_df.columns]
summary = pd.DataFrame({
    "1 token (entero)": [(cov_df[c] == 1).sum() for c in tok_cols],
    "2 piezas": [(cov_df[c] == 2).sum() for c in tok_cols],
    "3+ piezas": [(cov_df[c] >= 3).sum() for c in tok_cols],
    "media piezas/término": [round(cov_df[c].mean(), 2) for c in tok_cols],
}, index=tok_cols)
print("Resumen sobre {} términos clave:".format(len(KEY_TERMS)))
print(summary.to_string())
print()
print(cov_df.to_string())

In [ ]:
# ── Visualización de la cobertura ────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(15, 7))

cov_plot = cov_df.sort_values(tok_cols[0])
cov_plot.plot.barh(ax=axes[0], color=["seagreen", "indianred"][:len(tok_cols)])
axes[0].axvline(1, color="black", ls=":", lw=1)
axes[0].text(1.05, -0.8, "ideal: 1 token", fontsize=9)
axes[0].set_title("Piezas por término técnico")
axes[0].set_xlabel("número de tokens")

summary[["1 token (entero)", "2 piezas", "3+ piezas"]].plot.bar(
    ax=axes[1], stacked=True, color=["seagreen", "gold", "indianred"], rot=0)
axes[1].set_title("Términos enteros vs fragmentados")
axes[1].set_ylabel("número de términos")
plt.tight_layout()
plt.show()

if gpt2_tok is not None:
    ex = "subestación"
    print("Ejemplo '{}': propio = {} | GPT-2 = {}".format(
        ex, own_tok.encode(" " + ex).tokens, gpt2_tok.tokenize(" " + ex)))
    print("\nMoraleja: el tokenizador se entrena con EL corpus del modelo, no se hereda")
    print("a ciegas — 30 min de entrenamiento de tokenizador valen semanas de GPU.")

<a id="sec13"></a>
## 13. Distribución de longitudes de secuencia

Última pregunta operativa antes de entrenar: ¿cómo se reparten los **tokens por documento**? Esta distribución gobierna dos decisiones de la Fase 3 (pre-entrenamiento):

1. **`max_seq_len`** — si la mayoría de los documentos miden 80–120 tokens y la ventana es de 256, entrenar con un documento por secuencia desperdicia >50% de cada batch en `<pad>` (posiciones que cuestan FLOPs y no aportan gradiente).
2. **Packing** — la solución estándar: concatenar documentos separados por `<|endoftext|>` y trocear el stream en ventanas exactas de `max_seq_len`. Aprovechamiento ~100%, al precio de que algunas secuencias mezclen el final de un documento con el inicio del siguiente (el token separador le enseña al modelo dónde "resetear" el contexto).

$$\text{aprovechamiento}_{sin\,packing} = \frac{\sum_i \min(T_i, L)}{N \cdot L} \qquad \text{vs} \qquad \text{aprovechamiento}_{packing} \approx 1$$

In [ ]:
# ── Histograma de tokens/documento e implicaciones de packing ────────
doc_lens = np.array([len(own_tok.encode(t).ids) for t in ALL_TEXTS])
L = CONFIG["max_seq_len"]

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(doc_lens, bins=15, color="steelblue", edgecolor="white")
ax.axvline(L, color="red", ls="--", lw=2, label="max_seq_len = {} ({})".format(L, PROFILE))
ax.axvline(np.median(doc_lens), color="darkorange", ls=":", lw=2,
           label="mediana = {:.0f}".format(np.median(doc_lens)))
ax.set_xlabel("tokens por documento")
ax.set_ylabel("número de documentos")
ax.set_title("Distribución de longitudes del mini-corpus (tokenizador propio)")
ax.legend()
plt.tight_layout()
plt.show()

print("Percentiles de tokens/doc:  p50 = {:.0f} | p90 = {:.0f} | p99 = {:.0f} | max = {}".format(
    *np.percentile(doc_lens, [50, 90, 99]), doc_lens.max()))

# Aprovechamiento del batch: 1 doc/secuencia vs packing
util_sin = np.minimum(doc_lens, L).sum() / (len(doc_lens) * L)
total_tokens = doc_lens.sum() + len(doc_lens)            # +1 <|endoftext|> por doc
n_seqs_packed = math.ceil(total_tokens / L)
print("\nSin packing : {} secuencias, aprovechamiento = {:.0%} (el resto es <pad>)".format(
    len(doc_lens), util_sin))
print("Con packing : {} secuencias, aprovechamiento ≈ 100%".format(n_seqs_packed))
print("-> Con esta distribución, packing reduce {:.1f}x el cómputo por época.".format(
    len(doc_lens) / n_seqs_packed * util_sin / util_sin if n_seqs_packed else 0) if False else
    "-> Packing usa {:.1f}x menos secuencias para ver los mismos tokens.".format(
    len(doc_lens) / max(1, n_seqs_packed)))

<a id="sec14"></a>
## 14. Resumen

| Concepto | Idea clave | Dónde se usa |
|---|---|---|
| Vocabulario abierto | Palabras = OOV inevitable (Heaps); caracteres = $T$ explota; subpalabras = ambos costos acotados | Todos los LLM modernos |
| **BPE** (§2) | Greedy: fusionar el par más frecuente maximiza la compresión inmediata; el orden de merges ES el modelo | GPT-2/3/4, Llama, nuestro lab |
| **Byte-level** (§3) | Alfabeto = 256 bytes ⇒ `<UNK>` imposible; biyección bytes→unicode imprimible (`Ġ`) | GPT-2+, nuestro tokenizador |
| **WordPiece** (§4) | Fusiona por $\frac{c(a,b)}{c(a)c(b)}$ (PMI ≈ ganancia de verosimilitud), prefijo `##` | BERT y familia |
| **SentencePiece** (§5) | El espacio es un carácter más (`▁`): round-trip exacto, clave para código e idiomas sin espacios | T5, Llama 1/2, Mistral |
| **Unigram LM** (§6) | Segmentación como inferencia: $P(\mathbf{s})=\prod p(t_i)$, Viterbi + EM + poda | SentencePiece (default) |
| Fertilidad (§7) | tokens/palabra: mide el "costo de contexto" por idioma/dominio — refleja el corpus de entrenamiento del tokenizador | Auditoría de tokenizadores |
| Zipf + entropía (§10) | $f \propto r^{-s}$; $H \ll \log_2 V$: el unigrama es la cota trivial del loss | Sanidad del corpus tokenizado |
| Compresión (§11) | chars/token fija el contexto efectivo y el costo por época | Decisión de `vocab_size` |
| Packing (§13) | Concatenar docs con `<|endoftext|>` ⇒ aprovechamiento ~100% del batch | Fase 3 (pre-entrenamiento) |

**Artefacto producido:** `tokenizer/trained/tokenizer_nb02_{profile}.json` — ByteLevel-BPE con NFKC, round-trip verificado. En la Fase 1 real se re-entrena con el corpus completo de 21 dominios; el notebook `03` lo usará para construir los embeddings.

## Ejercicios propuestos

1. **[Básico]** Re-entrena el BPE de §2 con `num_merges = 50` y `500`. Grafica la fertilidad media del corpus en función de $k$ y localiza el "codo" donde agregar vocabulario deja de pagar. ¿Coincide con la curva de compresión de §2?

2. **[Básico]** En §4 el criterio WordPiece prefirió pares raros pero exclusivos. Construye un mini-corpus adversarial de 5 palabras donde BPE y WordPiece elijan un **primer merge distinto**, y verifica tu predicción a mano ejecutando ambos.

3. **[Intermedio]** Implementa `bpe_encode_word_fast` que en lugar de iterar todos los merges en orden use un dict `{par: prioridad}` y una cola de prioridad para aplicar siempre el merge de menor rango presente en la palabra (así lo hace la librería real). Verifica con `assert` que produce exactamente las mismas segmentaciones que la versión de §2 sobre las 50 palabras más frecuentes del corpus.

4. **[Intermedio]** El Viterbi de §6 devuelve solo la mejor segmentación. Modifícalo para devolver las **top-3** segmentaciones con sus log-probs (pista: guarda en cada posición los 3 mejores estados). Úsalo para implementar *subword regularization* casera: muestrear entre las top-3 con probabilidad proporcional a $P(\mathbf{s})^{\alpha}$, $\alpha = 0.5$.

5. **[Avanzado]** Entrena dos tokenizadores propios de §8: uno solo con los documentos en español y otro solo en inglés, con el mismo `vocab_size`. Mide la fertilidad *cruzada* (tokenizador ES sobre texto EN y viceversa) y la cobertura de §12 para cada uno. ¿Cuánto cuesta, en fertilidad, "prestar" el tokenizador del otro idioma? Relaciona el resultado con el heatmap de §7.

## Referencias

**Papers fundacionales**

- Sennrich, Haddow & Birch (2016). *Neural Machine Translation of Rare Words with Subword Units* — el paper que trajo BPE al NLP. [arXiv:1508.07909](https://arxiv.org/abs/1508.07909)
- Kudo (2018). *Subword Regularization: Improving Neural Network Translation Models with Multiple Subword Candidates* — el modelo Unigram LM (§6). [arXiv:1804.10959](https://arxiv.org/abs/1804.10959)
- Kudo & Richardson (2018). *SentencePiece: A simple and language independent subword tokenizer* (§5). [arXiv:1808.06226](https://arxiv.org/abs/1808.06226)
- Radford et al. (2019). *Language Models are Unsupervised Multitask Learners* (GPT-2) — sección 2.2: byte-level BPE (§3). [PDF](https://cdn.openai.com/better-language-models/language_models_are_unsupervised_multitask_learners.pdf)
- Schuster & Nakajima (2012). *Japanese and Korean Voice Search* — el WordPiece original (§4). [IEEE](https://ieeexplore.ieee.org/document/6289079)
- Gage (1994). *A New Algorithm for Data Compression* — el BPE original, como compresor.

**Código y cursos**

- Karpathy — [minbpe](https://github.com/karpathy/minbpe): BPE byte-level mínimo y limpio; y su video [*Let's build the GPT Tokenizer*](https://www.youtube.com/watch?v=zduSFxRajkE).
- HuggingFace — [tokenizers](https://github.com/huggingface/tokenizers) (la librería de §8) y el [capítulo 6 del curso de HF](https://huggingface.co/learn/nlp-course/chapter6/1) (entrenar tokenizadores).
- Google — [sentencepiece](https://github.com/google/sentencepiece).

**Lecturas complementarias**

- Rust et al. (2021). *How Good is Your Tokenizer? On the Monolingual Performance of Multilingual Language Models* — fertilidad por idioma, base de §7. [arXiv:2012.15613](https://arxiv.org/abs/2012.15613)
- Zouhar et al. (2023). *Tokenization and the Noiseless Channel* — teoría de la información aplicada a tokenizadores (§10–11). [arXiv:2306.16842](https://arxiv.org/abs/2306.16842)

---

**Siguiente notebook:** `03_embeddings.ipynb` — de ids discretos a vectores densos: la tabla $W_E \in \mathbb{R}^{V \times d_{model}}$ que este tokenizador acaba de dimensionar.